In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch1. NLTK 자연어 처리</font>

- 자연어 처리 : 텍스트 전처리, 단어의 빈도수 측정, 문서 유사도 측정, 연관 분석, 딥러닝, 워드 임베딩, 텍스트 분류, 어휘 데이터 베이스 사용, 감정 분석, 분류 분석


# 1. NLTK pkg
- 텍스트 전처리, 단어 빈도수 측정 : 토큰화 (문장, 어절, 형태소 단위로 분리), 정규표현식을 이용한 불용어 처리, 어간 추출
- 품사 태깅 : 단어별 품사를 식별
```
pip install nltk==3.7
```

- sent_tokenize() : 문장 단위로 분리한 list 반환
- word_tokenize() : 단어 단위로 분리한 list 반환
- RegexpTokenizer class : 정규표현식을 이용한 토큰화 작업

In [ ]:
import nltk

In [ ]:
# 말뭉치 다운로드
nltk.download()

In [ ]:
# 말뭉치 리스트
nltk.corpus.gutenberg.fileids()

In [ ]:
emma = nltk.corpus.gutenberg.raw('austen-emma.txt')
print(emma[:45])
print('type :', type(emma))
print('글자 수 :',len(emma))

In [ ]:
# sent_tokenize() : 문장 단위로 분리한 list 반환
from nltk.tokenize import sent_tokenize
sent_tokens = sent_tokenize(emma)
print('문장 수', len(sent_tokens))
print('첫 문장 :', sent_tokens[0])

In [ ]:
# word_tokenize() : 단어 단위로 분리한 list 반환
from nltk.tokenize import word_tokenize
print(word_tokenize(sent_tokens[0]))

In [ ]:
# RegexpTokenizer class : 정규표현식을 이용한 토큰화 작업
from nltk.tokenize import RegexpTokenizer
ret = RegexpTokenizer('\d+') # [0-9]+
digits = ret.tokenize(emma)
print(digits)
ret1 = RegexpTokenizer('\w+') # [a-zA-Z0-9]+
words = ret1.tokenize(sent_tokens[0])
print(words)
ret2 = RegexpTokenizer('[a-zA-Z]+')
words2 = ret2.tokenize(sent_tokens[0])
print(words2)

# 2. 형태소 분석
- 자연어 처리의 기본 : 형태소 분석, 품사 태깅
    * 어간 추출 (stemmer)
        - PorterStemmer, LancasterStemmer, RegexpStemmer
    * 품사 태깅 (pos_tag)

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer
words = ['sending','cooking','files','lives','crying','dying']

# 어간 추출 : 1. PorterStemmer
pst = PorterStemmer()
print([pst.stem(word) for word in words])

# 어간 추출 : 2. LancasterStemmer
lst = LancasterStemmer()
print([lst.stem(word) for word in words])

# 어간 추출 : 3. RegexpStemmer
rst = RegexpStemmer('ing')
print([rst.stem(word) for word in words])

# 3. 품사 태깅

In [ ]:
from nltk.tag import pos_tag
tagged_list = pos_tag(word_tokenize(sent_tokens[0]))
print('품사 태깅 결과 :', tagged_list)

In [ ]:
# 명사만 추출 : NN, NNP, NNS, NNPS
print([word for word, tag in tagged_list if tag.startswith('NN')])

## Quiz. in emma

1. 특수문자가 들어가지 않은 3글자 이상의 단어만 추출하여 품사 태깅
2. 'Emma' 단어가 등장한 횟수와 해당 단어의 품사 종류 출력
3. 특정 품사(명사)의 단어 종류 개수 출력

In [ ]:
# 1
emma = nltk.corpus.gutenberg.raw('austen-emma.txt')

ret = RegexpTokenizer('\w{3,}')
words = ret.tokenize(emma)

tagged_list = pos_tag(words)

In [ ]:
# 2
count = 0
emma_tags = set()
for word, tag in tagged_list:
    if word=='Emma':
        count += 1
        emma_tags.add(tag)
print(count)
print(emma_tags)

In [ ]:
# 2
tags = [tag for word, tag in tagged_list if word=='Emma']
cnt = len(tags)
pos = set(tags)
print('Emma 등장 횟수 :', cnt)
print('분류된 품사들 :', pos)

In [ ]:
# 2-1 Emma를 NNP, NNS 등으로 태깅한 횟수(Series이용)
import pandas as pd
pos_cnt = pd.Series([0]*len(pos), index=pos)
for word, tag in tagged_list:
    if word == 'Emma':
        pos_cnt[tag] += 1
pos_cnt.sort_values(ascending=False) # 내림차순

In [ ]:
# 2-1  Emma를 NNP, NNS 등으로 태깅한 횟수(dict이용)
pos_cnt = dict()
for word, tag in tagged_list:
    if word == 'Emma':
        if tag in pos_cnt:
            pos_cnt[tag] += 1
        else:
            pos_cnt[tag] = 1
print(pos_cnt)

In [ ]:
# 2-1 
sorted(pos_cnt.items(), key=lambda x:x[1], reverse=True)

In [ ]:
# 3
len(set([word for word, tag in tagged_list if tag.startswith('NN')]))

In [ ]:
# 3
noun_tags = ["NN", "NNS", "NNP", "NNPS"]
nouns_list = []
for word, tag in tagged_list:
    if tag in  noun_tags:
        nouns_list.append(word)
nouns_list = [word for word, tag in tagged_list if tag in noun_tags]
print('명사가 나온 횟수 :', len(nouns_list))
print('출현한 명사 수(반복제거) : ', len(set(nouns_list)))
print('한단어가 나오는 평균빈도수 :',
     len(nouns_list) / len(set(nouns_list)))

# 4. 최빈 단어 시각화

In [ ]:
print(nouns_list[:10])

In [ ]:
import matplotlib.pyplot as plt
from nltk import Text
# Text : 단어 빈도 분석, 단어 검색 클래스
emma_text = Text(nouns_list)
plt.figure(figsize=(18,4))
emma_text.plot(20)
plt.show()

In [ ]:
freqDist = emma_text.vocab()
# freqDist
most_common = freqDist.most_common(20) # 상위 20개
most_common
xlabel = [word for word, count in most_common]
yvalue = [count for word, count in most_common]
plt.figure(figsize=(18,4))
plt.plot(xlabel, yvalue, marker=None)
#plt.xticks(rotation=45)
plt.xlabel('Samples')
plt.ylabel('Counts')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
emma_text.count('Emma')

In [ ]:
# Emma, Harriet, Jane, Frank 각각 단어가 emma 소설 전체에서 등장하는 위치
plt.figure(figsize=(20,4))
emma_text.dispersion_plot(['Emma', 'Harriet','Jane', 'Frank'])

In [ ]:
from nltk import FreqDist
tags = ['NNP', 'NNPS']
name_list = [word for word, tag in tagged_list if tag in tags]
name_count = FreqDist(name_list)
name_count

In [ ]:
for name, count in name_count.items():
    if count>150:
        print(name, ':', count)

In [ ]:
name_count_s = pd.Series(name_count)
name_count_s.sort_values(ascending=False, inplace=True)
name_count_s[name_count_s>150]